# Unidad 1: Arquitecturas Big Data y ETL batch distribuido

## Sesión 4: ML distribuido con Spark MLlib — Regresión

### Objetivo
Integrar y validar datos de varias fuentes, preparar una salida
analítica y comparar modelos de regresión mediante métricas.

### Práctica guiada
Utilizaremos las fuentes indicadas por el docente:
- campo_electrico.csv
- campo_magnetico.csv
- variables_ambientales.csv

### Variable a estimar
Estimaremos CE, el campo eléctrico, a partir de otras
variables medidas en el mismo instante.

Este ejercicio no es un pronóstico de valores futuros.

### Modelos y evaluación
Compararemos regresión lineal, configuraciones de
regularización y Random Forest mediante RMSE, R² y MAE.

### Aplicación al Proyecto Sello
Después aplicaremos el flujo de preparación, entrenamiento
y evaluación a AgroSmart Puno.

### Estado de los datos
Los tres CSV de la práctica guiada todavía deben incorporarse
a la carpeta data de esta sesión.

### Crear la sesión de Spark

Iniciaremos Spark con la configuración indicada por el docente:
- Aplicación: sesion4-ml-distribuido-regresion.
- Ejecución local con los núcleos disponibles.
- Puerto interno de Spark UI: 4040.
- Ocho particiones de shuffle.
- Cuatro GB de memoria para el driver.

También definiremos la ubicación de los datos de S04.
La ruta de resultados se declarará cuando corresponda escribirlos.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 19:41:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"

## Fase 1 — Business Understanding

### Variable numérica a estimar

El objetivo de la práctica guiada es estimar CE,
el valor del campo eléctrico, a partir de otras variables
medidas en el mismo instante.

CE será la variable objetivo. Las variables utilizadas
para estimarla serán los predictores.

Este ejercicio es de regresión: produce una estimación
numérica. No pronostica valores futuros ni utiliza
un historial temporal como entrada.

### Decisión y alcance de la comparación

Evaluaremos si las variables disponibles permiten estimar CE
con un desempeño útil para este ejercicio.

Compararemos:
- Un modelo base de regresión lineal.
- Tres configuraciones de regularización.
- Un segundo algoritmo: Random Forest.

Utilizaremos las mismas métricas de evaluación:
RMSE, R² y MAE.

La comparación permitirá seleccionar el modelo con mejor
desempeño según el criterio documentado. No realizaremos
una búsqueda exhaustiva de hiperparámetros.

Sin un umbral de error aceptable para una aplicación concreta,
las métricas no bastan para afirmar que el modelo está
preparado para uso operativo.

## Actividad autónoma — AgroSmart Puno

### Fuente y objetivo

Utilizaremos los datos históricos de Open-Meteo conservados
en S02 para Azángaro, correspondientes a junio de 2025.

Prepararemos un CSV integrado con variables de atmósfera
y suelo, conservando la procedencia de la descarga.

El objetivo será estimar la temperatura del aire mediante
otras variables del mismo instante. No será un pronóstico
de heladas futuras.

Esta actividad corresponde al Proyecto Sello y no sustituye
la práctica guiada de campo eléctrico y magnético.

In [3]:
from pathlib import Path

carpeta_fuente = Path("/opt/s02-fundamentos/data/agrosmart")

archivos_agro = sorted(
    carpeta_fuente.glob("open_meteo_azangaro_*.json")
)

print("Archivos encontrados:", len(archivos_agro))

for archivo in archivos_agro:
    print(archivo.name)

Archivos encontrados: 1
open_meteo_azangaro_20261116T210745703464Z.json


### Verificar la fuente antes de generar el CSV

Leeremos la descarga original y revisaremos su procedencia,
periodo y variables horarias.

Comprobaremos que las listas tengan la misma longitud
para unir correctamente los valores de cada hora.

Si el periodo no corresponde a junio de 2025,
detendremos este paso.

In [4]:
import json

if not archivos_agro:
    raise FileNotFoundError(
        "No se encontró la descarga de AgroSmart en S02."
    )

ruta_fuente_agro = archivos_agro[-1]

with ruta_fuente_agro.open("r", encoding="utf-8") as archivo:
    original_agro = json.load(archivo)

horarios = original_agro["respuesta"]["hourly"]

variables = [
    "time",
    "temperature_2m",
    "relative_humidity_2m",
    "soil_temperature_0_to_7cm",
    "soil_moisture_0_to_7cm",
]

for nombre in variables:
    if nombre not in horarios:
        raise ValueError(f"Falta la variable: {nombre}")
    if not isinstance(horarios[nombre], list):
        raise TypeError(f"{nombre} debe ser una lista.")

longitudes = {nombre: len(horarios[nombre]) for nombre in variables}

if len(set(longitudes.values())) != 1 or not horarios["time"]:
    raise ValueError(f"Listas vacías o desiguales: {longitudes}")

if (
    horarios["time"][0] != "2025-06-01T00:00"
    or horarios["time"][-1] != "2025-06-30T23:00"
    or longitudes["time"] != 720
):
    raise ValueError("El periodo no coincide con junio de 2025 completo.")

print("Fuente:", original_agro.get("fuente"))
print("Archivo:", ruta_fuente_agro.name)
print("Periodo:", horarios["time"][0], "a", horarios["time"][-1])
print("Registros:", longitudes["time"])
print("Unidades:", original_agro["respuesta"].get("hourly_units"))

Fuente: Open-Meteo Historical Weather API
Archivo: open_meteo_azangaro_20261116T210745703464Z.json
Periodo: 2025-06-01T00:00 a 2025-06-30T23:00
Registros: 720
Unidades: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'soil_temperature_0_to_7cm': '°C', 'soil_moisture_0_to_7cm': 'm³/m³'}


### Crear el CSV integrado de AgroSmart

Generaremos una fila por hora con las variables de atmósfera
y suelo de junio de 2025.

Los valores nulos se escribirán como campos vacíos,
sin reemplazarlos por cero.

Conservaremos también una copia del JSON original
para documentar la procedencia.

In [5]:
import csv
import shutil

carpeta_agro_s04 = Path(ORIGEN_DATOS) / "agrosmart"
carpeta_agro_s04.mkdir(parents=True, exist_ok=True)

ruta_csv_agro = carpeta_agro_s04 / "agrosmart_azangaro_2025_06.csv"

columnas_csv = [
    "fecha_hora",
    "localidad",
    "temperatura_aire_c",
    "humedad_relativa_pct",
    "temperatura_suelo_c",
    "humedad_suelo_m3_m3",
]

# Reemplaza este CSV si volvemos a ejecutar la celda.
with ruta_csv_agro.open("w", newline="", encoding="utf-8") as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(columnas_csv)

    for i, fecha in enumerate(horarios["time"]):
        escritor.writerow([
            fecha,
            "Azangaro",
            horarios["temperature_2m"][i],
            horarios["relative_humidity_2m"][i],
            horarios["soil_temperature_0_to_7cm"][i],
            horarios["soil_moisture_0_to_7cm"][i],
        ])

ruta_respaldo = carpeta_agro_s04 / ruta_fuente_agro.name

if not ruta_respaldo.exists():
    shutil.copy2(ruta_fuente_agro, ruta_respaldo)

with ruta_csv_agro.open(newline="", encoding="utf-8") as archivo:
    filas_guardadas = sum(1 for _ in csv.DictReader(archivo))

assert filas_guardadas == len(horarios["time"])

print("CSV creado:", ruta_csv_agro)
print("Registros guardados:", filas_guardadas)
print("JSON de procedencia:", ruta_respaldo)


CSV creado: /opt/s04-ml-distribuido-regresion/data/agrosmart/agrosmart_azangaro_2025_06.csv
Registros guardados: 720
JSON de procedencia: /opt/s04-ml-distribuido-regresion/data/agrosmart/open_meteo_azangaro_20261116T210745703464Z.json


### Cargar el CSV con esquema explícito

Definiremos fecha_hora y localidad como texto,
y las cuatro variables ambientales como números decimales.

Permitiremos valores nulos para detectarlos antes
del entrenamiento. Después revisaremos el esquema,
las primeras filas y la cantidad de registros.

In [6]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema_agro_ml = StructType([
    StructField("fecha_hora", StringType(), True),
    StructField("localidad", StringType(), True),
    StructField("temperatura_aire_c", DoubleType(), True),
    StructField("humedad_relativa_pct", DoubleType(), True),
    StructField("temperatura_suelo_c", DoubleType(), True),
    StructField("humedad_suelo_m3_m3", DoubleType(), True),
])

df_agro_ml = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema(schema_agro_ml)
    .csv(str(ruta_csv_agro))
)

df_agro_ml.printSchema()
df_agro_ml.show(5, truncate=False)

print("Registros cargados:", df_agro_ml.count())

root
 |-- fecha_hora: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- temperatura_aire_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- temperatura_suelo_c: double (nullable = true)
 |-- humedad_suelo_m3_m3: double (nullable = true)

+----------------+---------+------------------+--------------------+-------------------+-------------------+
|fecha_hora      |localidad|temperatura_aire_c|humedad_relativa_pct|temperatura_suelo_c|humedad_suelo_m3_m3|
+----------------+---------+------------------+--------------------+-------------------+-------------------+
|2025-06-01T00:00|Azangaro |2.9               |70.0                |4.3                |0.212              |
|2025-06-01T01:00|Azangaro |2.9               |70.0                |4.0                |0.212              |
|2025-06-01T02:00|Azangaro |2.8               |70.0                |3.8                |0.212              |
|2025-06-01T03:00|Azangaro |2.1               |74.0  

### Diagnóstico de calidad para el modelado

Comprobaremos:
- Valores nulos por columna.
- Fechas que no pueden interpretarse.
- Repeticiones de localidad y hora.
- Humedades fuera de sus rangos básicos.

La temperatura del aire será la variable objetivo.
No inventaremos valores faltantes para entrenar el modelo.

In [7]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Lima")

df_agro_revision_ml = df_agro_ml.withColumn(
    "fecha_hora_ts",
    F.try_to_timestamp(
        F.col("fecha_hora"),
        F.lit("yyyy-MM-dd'T'HH:mm")
    )
)

# Nulos por columna, incluida la fecha convertida
df_agro_revision_ml.select([
    F.count(
        F.when(F.col(c).isNull(), 1)
    ).alias(f"nulos_{c}")
    for c in df_agro_revision_ml.columns
]).show(vertical=True)

# Claves repetidas
df_agro_revision_ml.groupBy(
    "localidad", "fecha_hora_ts"
).count().filter(
    F.col("count") > 1
).show(truncate=False)

# Humedades fuera de rango; los nulos se cuentan arriba
df_agro_revision_ml.select(
    F.count(
        F.when(
            ~F.col("humedad_relativa_pct").between(0, 100), 1
        )
    ).alias("humedad_aire_fuera_rango"),
    F.count(
        F.when(
            ~F.col("humedad_suelo_m3_m3").between(0, 1), 1
        )
    ).alias("humedad_suelo_fuera_rango")
).show()

-RECORD 0-------------------------
 nulos_fecha_hora           | 0   
 nulos_localidad            | 0   
 nulos_temperatura_aire_c   | 0   
 nulos_humedad_relativa_pct | 0   
 nulos_temperatura_suelo_c  | 0   
 nulos_humedad_suelo_m3_m3  | 0   
 nulos_fecha_hora_ts        | 0   

+---------+-------------+-----+
|localidad|fecha_hora_ts|count|
+---------+-------------+-----+
+---------+-------------+-----+

+------------------------+-------------------------+
|humedad_aire_fuera_rango|humedad_suelo_fuera_rango|
+------------------------+-------------------------+
|                       0|                        0|
+------------------------+-------------------------+



### Preparar los datos Silver para regresión

Utilizaremos:
- Objetivo: temperatura_aire_c.
- Predictores: humedad_relativa_pct, temperatura_suelo_c
  y humedad_suelo_m3_m3.

Conservaremos registros con fecha y localidad válidas,
variables numéricas disponibles y humedades dentro de rango.

No reemplazaremos temperaturas faltantes por cero.
Eliminaremos filas completamente idénticas y detendremos
el proceso si existen valores distintos para una misma
localidad y hora.

In [8]:
objetivo = "temperatura_aire_c"

predictores = [
    "humedad_relativa_pct",
    "temperatura_suelo_c",
    "humedad_suelo_m3_m3",
]

df_base_ml = df_agro_revision_ml.withColumn(
    "localidad",
    F.trim(F.col("localidad"))
)

condicion_ml = (
    F.col("localidad").isNotNull()
    & (F.col("localidad") != "")
    & F.col("fecha_hora_ts").isNotNull()
)

for nombre in [objetivo] + predictores:
    condicion_ml = (
        condicion_ml
        & F.col(nombre).isNotNull()
        & ~F.isnan(F.col(nombre))
        & (F.abs(F.col(nombre)) < float("inf"))
    )

condicion_ml = (
    condicion_ml
    & F.col("humedad_relativa_pct").between(0, 100)
    & F.col("humedad_suelo_m3_m3").between(0, 1)
)

df_agro_excluidos_ml = df_base_ml.filter(~condicion_ml)

df_agro_silver_ml = (
    df_base_ml
    .filter(condicion_ml)
    .dropDuplicates()
)

conflictos_ml = (
    df_agro_silver_ml
    .groupBy("localidad", "fecha_hora_ts")
    .count()
    .filter(F.col("count") > 1)
)

print("Originales:", df_agro_ml.count())
print("Excluidos por calidad:", df_agro_excluidos_ml.count())
print("Registros Silver:", df_agro_silver_ml.count())

if conflictos_ml.count() > 0:
    conflictos_ml.show(truncate=False)
    raise ValueError("Hay registros diferentes para una misma localidad y hora.")

if df_agro_silver_ml.count() == 0:
    raise ValueError("No quedaron registros para entrenar.")

Originales: 720
Excluidos por calidad: 0
Registros Silver: 720


### Preparar y guardar la salida para modelado

Seleccionaremos la fecha, localidad, variable objetivo
y predictores que utilizará el modelo.

Añadiremos año y mes para organizar la salida Parquet.
Esta tabla será la entrada Gold para el modelado,
derivada de los registros Silver validados.

Después leeremos la salida y comprobaremos su conteo.
La muestra contiene únicamente junio de 2025.

In [9]:
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"

df_agro_gold_ml = (
    df_agro_silver_ml
    .select(
        "fecha_hora_ts",
        "localidad",
        objetivo,
        *predictores
    )
    .withColumn("anio", F.year("fecha_hora_ts"))
    .withColumn("mes", F.month("fecha_hora_ts"))
)

RUTA_GOLD_ML = f"{ARTIFACTS}/agrosmart/gold_modelado"

(
    df_agro_gold_ml
    .write
    .mode("overwrite")
    .partitionBy("anio", "mes")
    .parquet(RUTA_GOLD_ML)
)

df_modelado = spark.read.parquet(RUTA_GOLD_ML)

filas_antes = df_agro_gold_ml.count()
filas_leidas = df_modelado.count()

print("Filas antes de guardar:", filas_antes)
print("Filas recuperadas:", filas_leidas)

assert filas_antes == filas_leidas, "Los conteos no coinciden."

df_modelado.printSchema()

Filas antes de guardar: 720
Filas recuperadas: 720
root
 |-- fecha_hora_ts: timestamp (nullable = true)
 |-- localidad: string (nullable = true)
 |-- temperatura_aire_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- temperatura_suelo_c: double (nullable = true)
 |-- humedad_suelo_m3_m3: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)



### Construir el vector de predictores

Agruparemos tres variables en una columna llamada features:
- Humedad relativa del aire.
- Temperatura del suelo.
- Humedad del suelo.

La temperatura del aire será la variable objetivo
y no se incluirá en features.

El modelo estimará una medición del mismo instante;
no será un pronóstico de temperatura futura.

In [10]:
from pyspark.ml.feature import VectorAssembler

ensamblador = VectorAssembler(
    inputCols=predictores,
    outputCol="features",
    handleInvalid="error"
)

df_features = ensamblador.transform(df_modelado)

df_features.select(
    "fecha_hora_ts",
    objetivo,
    "features"
).show(5, truncate=False)

+-------------------+------------------+-----------------+
|fecha_hora_ts      |temperatura_aire_c|features         |
+-------------------+------------------+-----------------+
|2025-06-01 04:00:00|1.9               |[76.0,3.5,0.212] |
|2025-06-01 11:00:00|11.1              |[44.0,11.9,0.212]|
|2025-06-01 17:00:00|10.0              |[43.0,12.4,0.209]|
|2025-06-02 06:00:00|-0.4              |[99.0,2.9,0.208] |
|2025-06-02 13:00:00|12.7              |[31.0,15.4,0.205]|
+-------------------+------------------+-----------------+
only showing top 5 rows


### Separar entrenamiento, validación y prueba

Utilizaremos tres periodos:
- Entrenamiento: del 1 al 18 de junio de 2025.
- Validación: del 19 al 24 de junio.
- Prueba final: del 25 al 30 de junio.

Entrenaremos los modelos con el primer bloque y elegiremos
la configuración mediante validación.

Reservaremos la prueba para evaluar el modelo seleccionado,
sin utilizarla para elegir configuraciones.

Esta separación temporal no convierte el ejercicio
en un pronóstico: los predictores son del mismo instante.

In [11]:
fecha_registro = F.to_date(F.col("fecha_hora_ts"))

df_train = df_features.filter(
    fecha_registro < F.lit("2025-06-19").cast("date")
)

df_validacion = df_features.filter(
    (fecha_registro >= F.lit("2025-06-19").cast("date"))
    & (fecha_registro < F.lit("2025-06-25").cast("date"))
)

df_test = df_features.filter(
    fecha_registro >= F.lit("2025-06-25").cast("date")
)

n_train = df_train.count()
n_validacion = df_validacion.count()
n_test = df_test.count()

print("Entrenamiento:", n_train)
print("Validación:", n_validacion)
print("Prueba:", n_test)

assert min(n_train, n_validacion, n_test) > 0, (
    "Algún bloque quedó vacío."
)

assert n_train + n_validacion + n_test == df_features.count(), (
    "La separación no conserva el total de registros."
)

Entrenamiento: 432
Validación: 144
Prueba: 144


### Entrenar la regresión lineal base

El modelo aprenderá a estimar temperatura_aire_c mediante
las tres variables reunidas en features.

Utilizaremos únicamente df_train para ajustar el modelo.
La configuración base no aplicará regularización.

Después generaremos estimaciones sobre validación,
sin utilizar todavía la prueba final.

In [12]:
from pyspark.ml.regression import LinearRegression

regresion_base = LinearRegression(
    featuresCol="features",
    labelCol=objetivo,
    predictionCol="prediction",
    regParam=0.0,
    elasticNetParam=0.0,
    maxIter=100
)

modelo_base = regresion_base.fit(df_train)

predicciones_base = modelo_base.transform(df_validacion)

predicciones_base.select(
    "fecha_hora_ts",
    objetivo,
    "prediction"
).orderBy("fecha_hora_ts").show(5, truncate=False)

26/11/19 19:53:12 WARN Instrumentation: [88d4433f] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


+-------------------+------------------+------------------+
|fecha_hora_ts      |temperatura_aire_c|prediction        |
+-------------------+------------------+------------------+
|2025-06-19 00:00:00|3.1               |2.649823332907008 |
|2025-06-19 01:00:00|2.6               |2.0392646398030667|
|2025-06-19 02:00:00|1.8               |1.374679724170663 |
|2025-06-19 03:00:00|1.2               |0.9025241997763757|
|2025-06-19 04:00:00|0.4               |0.4594675893547736|
+-------------------+------------------+------------------+
only showing top 5 rows


### Evaluar la regresión lineal base

Calcularemos tres métricas sobre el bloque de validación:

- RMSE: error que penaliza especialmente las diferencias grandes.
- MAE: promedio del error absoluto.
- R²: compara el error cuadrático con la variación del objetivo.

RMSE y MAE se expresan en °C; cuanto menores, mejor.
R² puede ser negativo y no representa un porcentaje de exactitud.

No utilizaremos todavía la prueba final.

In [13]:
from pyspark.ml.evaluation import RegressionEvaluator

metricas_base = {}

for metrica in ["rmse", "r2", "mae"]:
    evaluador = RegressionEvaluator(
        labelCol=objetivo,
        predictionCol="prediction",
        metricName=metrica
    )

    metricas_base[metrica] = evaluador.evaluate(
        predicciones_base
    )

print("Resultados de validación:")
print(f"RMSE: {metricas_base['rmse']:.4f} °C")
print(f"R²:   {metricas_base['r2']:.4f}")
print(f"MAE:  {metricas_base['mae']:.4f} °C")

Resultados de validación:
RMSE: 1.7288 °C
R²:   0.9149
MAE:  1.2138 °C


### Comparar configuraciones de regularización

Entrenaremos tres alternativas sobre los mismos datos:
- L2: regParam=0.01, elasticNetParam=0.
- L1: regParam=0.01, elasticNetParam=1.
- Elastic Net: regParam=0.01, elasticNetParam=0.5.

regParam controla la intensidad de la penalización.
elasticNetParam determina la combinación de L1 y L2.

Compararemos RMSE, R² y MAE en validación.
El criterio principal será el menor RMSE.
La prueba final permanece reservada.

In [14]:
configuraciones = [
    ("L2", 0.01, 0.0),
    ("L1", 0.01, 1.0),
    ("ElasticNet", 0.01, 0.5),
]

modelos_comparados = {"Base": modelo_base}

resultados_modelos = [{
    "modelo": "Base",
    "rmse": metricas_base["rmse"],
    "r2": metricas_base["r2"],
    "mae": metricas_base["mae"],
}]

for nombre, penalizacion, mezcla in configuraciones:
    estimador = LinearRegression(
        featuresCol="features",
        labelCol=objetivo,
        predictionCol="prediction",
        regParam=penalizacion,
        elasticNetParam=mezcla,
        standardization=True,
        maxIter=100
    )

    modelo = estimador.fit(df_train)
    predicciones = modelo.transform(df_validacion)

    resultado = {"modelo": nombre}

    for metrica in ["rmse", "r2", "mae"]:
        resultado[metrica] = RegressionEvaluator(
            labelCol=objetivo,
            predictionCol="prediction",
            metricName=metrica
        ).evaluate(predicciones)

    modelos_comparados[nombre] = modelo
    resultados_modelos.append(resultado)

df_comparacion = spark.createDataFrame(resultados_modelos)

df_comparacion.select(
    "modelo", "rmse", "r2", "mae"
).orderBy("rmse").show(truncate=False)

[Stage 88:>                                                       (0 + 20) / 20]

+----------+------------------+------------------+------------------+
|modelo    |rmse              |r2                |mae               |
+----------+------------------+------------------+------------------+
|L2        |1.7281403002138047|0.9149318246207981|1.215485694194687 |
|Base      |1.7288404880436108|0.9148628767839107|1.2137677926952442|
|ElasticNet|1.7296642206688218|0.9147817276740803|1.2163068613926082|
|L1        |1.7312248388545897|0.9146278791043826|1.2172008682129265|
+----------+------------------+------------------+------------------+



### Comparar con Random Forest

Entrenaremos un modelo que combina varios árboles de decisión.

Utilizaremos los mismos predictores y bloques de entrenamiento
y validación que en la regresión lineal.

Compararemos RMSE, R² y MAE. Todavía no utilizaremos
la prueba final.

In [15]:
from pyspark.ml.regression import RandomForestRegressor

estimador_rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=objetivo,
    predictionCol="prediction",
    numTrees=50,
    maxDepth=5,
    seed=42
)

modelo_rf = estimador_rf.fit(df_train)

predicciones_rf = modelo_rf.transform(df_validacion)

resultado_rf = {"modelo": "RandomForest"}

for metrica in ["rmse", "r2", "mae"]:
    resultado_rf[metrica] = RegressionEvaluator(
        labelCol=objetivo,
        predictionCol="prediction",
        metricName=metrica
    ).evaluate(predicciones_rf)

modelos_comparados["RandomForest"] = modelo_rf

# Evita duplicar la fila si ejecutamos nuevamente esta celda.
resultados_modelos = [
    resultado
    for resultado in resultados_modelos
    if resultado["modelo"] != "RandomForest"
]
resultados_modelos.append(resultado_rf)

df_comparacion = spark.createDataFrame(resultados_modelos)

df_comparacion.select(
    "modelo", "rmse", "r2", "mae"
).orderBy("rmse").show(truncate=False)

+------------+------------------+------------------+------------------+
|modelo      |rmse              |r2                |mae               |
+------------+------------------+------------------+------------------+
|L2          |1.7281403002138047|0.9149318246207981|1.215485694194687 |
|Base        |1.7288404880436108|0.9148628767839107|1.2137677926952442|
|ElasticNet  |1.7296642206688218|0.9147817276740803|1.2163068613926082|
|L1          |1.7312248388545897|0.9146278791043826|1.2172008682129265|
|RandomForest|1.9681325360920976|0.8896638556925781|1.587322366432338 |
+------------+------------------+------------------+------------------+



### Importancia de los predictores

Consultaremos featureImportances del modelo Random Forest.

Relacionaremos cada importancia con el orden de las variables
utilizado por VectorAssembler.

Estos valores describen el uso de las variables en los árboles.
No demuestran causalidad ni indican el sentido de su efecto.


In [16]:
importancias = [
    (nombre, float(valor))
    for nombre, valor in zip(
        predictores,
        modelo_rf.featureImportances.toArray()
    )
]

df_importancias = spark.createDataFrame(
    importancias,
    ["predictor", "importancia"]
)

df_importancias.orderBy(
    F.col("importancia").desc()
).show(truncate=False)

+--------------------+-------------------+
|predictor           |importancia        |
+--------------------+-------------------+
|temperatura_suelo_c |0.5157240623812795 |
|humedad_relativa_pct|0.4367158839684089 |
|humedad_suelo_m3_m3 |0.04756005365031164|
+--------------------+-------------------+



### Seleccionar y evaluar el modelo final

Elegiremos el menor RMSE de validación.
Si hay empate, utilizaremos MAE y después el nombre
para mantener una regla reproducible.

Evaluaremos el modelo seleccionado en el bloque de prueba,
que no se utilizó para elegir configuraciones.

No cambiaremos de modelo según el resultado de prueba:
esas métricas serán la evaluación final de esta selección.

In [17]:
import math

resultados_validos = [
    resultado
    for resultado in resultados_modelos
    if all(
        math.isfinite(resultado[metrica])
        for metrica in ["rmse", "r2", "mae"]
    )
]

if not resultados_validos:
    raise ValueError("No hay modelos con métricas finitas.")

mejor_resultado = min(
    resultados_validos,
    key=lambda r: (r["rmse"], r["mae"], r["modelo"])
)

nombre_ganador = mejor_resultado["modelo"]
modelo_seleccionado = modelos_comparados[nombre_ganador]

print("Modelo seleccionado:", nombre_ganador)
print("RMSE de validación:", mejor_resultado["rmse"])

predicciones_test = modelo_seleccionado.transform(df_test)

metricas_test = {}

for metrica in ["rmse", "r2", "mae"]:
    metricas_test[metrica] = RegressionEvaluator(
        labelCol=objetivo,
        predictionCol="prediction",
        metricName=metrica
    ).evaluate(predicciones_test)

print("\nEvaluación final en prueba:")
print(f"RMSE: {metricas_test['rmse']:.4f} °C")
print(f"R²:   {metricas_test['r2']:.4f}")
print(f"MAE:  {metricas_test['mae']:.4f} °C")

Modelo seleccionado: L2
RMSE de validación: 1.7281403002138047

Evaluación final en prueba:
RMSE: 1.6536 °C
R²:   0.9084
MAE:  1.2131 °C


### Guardar el modelo seleccionado

Conservaremos el modelo elegido por RMSE de validación.

También guardaremos:
- Variable objetivo y orden de los predictores.
- Periodos de entrenamiento, validación y prueba.
- Métricas de comparación y evaluación final.
- Procedencia de los datos.

El modelo estima temperatura del mismo instante;
no es un pronóstico de heladas futuras.

In [18]:
from datetime import datetime, timezone

carpeta_modelo = Path(ARTIFACTS) / "agrosmart" / "modelo_seleccionado"
carpeta_modelo.mkdir(parents=True, exist_ok=True)

ruta_modelo = str(carpeta_modelo / "modelo")
ruta_ensamblador = str(carpeta_modelo / "ensamblador")

modelo_seleccionado.write().overwrite().save(ruta_modelo)
ensamblador.write().overwrite().save(ruta_ensamblador)

metadatos = {
    "modelo": nombre_ganador,
    "clase_modelo": type(modelo_seleccionado).__name__,
    "objetivo": objetivo,
    "unidad_objetivo": "°C",
    "predictores_en_orden": predictores,
    "zona_horaria": "America/Lima",
    "entrenamiento": ["2025-06-01", "2025-06-18"],
    "validacion": ["2025-06-19", "2025-06-24"],
    "prueba": ["2025-06-25", "2025-06-30"],
    "criterio_seleccion": "Menor RMSE de validación; desempate por MAE y nombre",
    "comparacion_validacion": resultados_modelos,
    "metricas_prueba": metricas_test,
    "archivo_fuente": ruta_fuente_agro.name,
    "fecha_guardado_utc": datetime.now(timezone.utc).isoformat(),
    "alcance": "Estimación contemporánea; no pronóstico futuro"
}

with (carpeta_modelo / "metadatos.json").open(
    "w", encoding="utf-8"
) as archivo:
    json.dump(
        metadatos,
        archivo,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    )

print("Modelo guardado:", ruta_modelo)
print("Ensamblador guardado:", ruta_ensamblador)
print("Metadatos:", carpeta_modelo / "metadatos.json")

Modelo guardado: /opt/s04-ml-distribuido-regresion/artifacts/agrosmart/modelo_seleccionado/modelo
Ensamblador guardado: /opt/s04-ml-distribuido-regresion/artifacts/agrosmart/modelo_seleccionado/ensamblador
Metadatos: /opt/s04-ml-distribuido-regresion/artifacts/agrosmart/modelo_seleccionado/metadatos.json


### Verificar la recuperación del modelo

Cargaremos desde disco el modelo y el ensamblador.

Generaremos predicciones sobre la misma prueba reservada
y comprobaremos que coincidan con las del modelo en memoria.

Esta verificación comprueba el guardado y recuperación;
no modifica la selección del modelo.

In [19]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import (
    LinearRegressionModel,
    RandomForestRegressionModel
)

ensamblador_recuperado = VectorAssembler.load(
    ruta_ensamblador
)

if nombre_ganador == "RandomForest":
    modelo_recuperado = RandomForestRegressionModel.load(ruta_modelo)
else:
    modelo_recuperado = LinearRegressionModel.load(ruta_modelo)

# Reconstruimos features usando el ensamblador guardado.
test_recuperado = ensamblador_recuperado.transform(
    df_test.drop("features")
)

predicciones_recuperadas = modelo_recuperado.transform(
    test_recuperado
)

comparacion_guardado = (
    predicciones_test
    .select(
        "localidad",
        "fecha_hora_ts",
        F.col("prediction").alias("prediccion_original")
    )
    .join(
        predicciones_recuperadas.select(
            "localidad",
            "fecha_hora_ts",
            F.col("prediction").alias("prediccion_recuperada")
        ),
        on=["localidad", "fecha_hora_ts"],
        how="inner"
    )
)

assert comparacion_guardado.count() == df_test.count()

diferencia_maxima = comparacion_guardado.select(
    F.max(
        F.abs(
            F.col("prediccion_original")
            - F.col("prediccion_recuperada")
        )
    ).alias("diferencia_maxima")
).first()["diferencia_maxima"]

print("Diferencia máxima:", diferencia_maxima)

assert diferencia_maxima is not None
assert diferencia_maxima <= 1e-9, (
    "Las predicciones recuperadas no coinciden."
)

print("Modelo recuperado y verificado correctamente.")

Diferencia máxima: 0.0
Modelo recuperado y verificado correctamente.


### Resumen de selección y evaluación

La configuración se seleccionó mediante el menor RMSE
de validación, con MAE como criterio de desempate.

Mostraremos las métricas del modelo seleccionado en
validación y prueba. RMSE y MAE se expresan en °C.

La prueba se utilizó después de seleccionar el modelo.

In [20]:
resumen_final = [
    (
        nombre_ganador,
        "Validación",
        mejor_resultado["rmse"],
        mejor_resultado["r2"],
        mejor_resultado["mae"]
    ),
    (
        nombre_ganador,
        "Prueba",
        metricas_test["rmse"],
        metricas_test["r2"],
        metricas_test["mae"]
    )
]

spark.createDataFrame(
    resumen_final,
    ["modelo", "bloque", "rmse_c", "r2", "mae_c"]
).show(truncate=False)

+------+----------+------------------+------------------+------------------+
|modelo|bloque    |rmse_c            |r2                |mae_c             |
+------+----------+------------------+------------------+------------------+
|L2    |Validación|1.7281403002138047|0.9149318246207981|1.215485694194687 |
|L2    |Prueba    |1.6536389951838601|0.9083958189999952|1.2131202136800352|
+------+----------+------------------+------------------+------------------+



### Resultados y límites del modelo

#### Objetivo
Estimar la temperatura del aire mediante humedad relativa,
temperatura del suelo y humedad del suelo del mismo instante.

#### Datos y separación
Utilicé datos de Open-Meteo para Azángaro, de junio de 2025.
Separé entrenamiento, validación y prueba por bloques de fechas.

#### Comparación
Comparé regresión lineal base, tres configuraciones
de regularización y Random Forest utilizando la misma
validación y las métricas RMSE, R² y MAE.

#### Resultados de mi ejecución
- Modelo seleccionado: Regresión lineal L2 (regParam=0.01, elasticNetParam=0).
- RMSE de validación: 1.7281403002 °C.
- RMSE de prueba: 1.6536389952 °C.
- MAE de prueba: 1.2131202137 °C.
- R² de prueba: 0.9083958190.
- Predictor con mayor importancia en Random Forest: temperatura_suelo_c (importancia aproximada: 0.5157).
- Verificación del modelo recuperado: correcta; diferencia máxima de predicciones: 0.0.

#### Limitaciones
La muestra contiene un solo mes y una ubicación.
Los predictores y el objetivo proceden de la misma fuente
meteorológica, por lo que esta prueba no valida el desempeño
frente a sensores independientes.

El modelo estima una variable del mismo instante.
No anticipa heladas futuras ni confirma daños agrícolas.

No se ha establecido un umbral de error aceptable
para utilizarlo en decisiones operativas.

### Hallazgo técnico: una mejora mínima no demuestra superioridad general

La regresión base obtuvo RMSE de validación 1.7288404880 °C y L2 obtuvo 1.7281403002 °C. La diferencia fue aproximadamente 0.00070019 °C, un margen muy pequeño. L2 fue seleccionada por la regla previamente definida, sin afirmar que esta diferencia sea estadísticamente significativa.

La base tuvo un MAE de validación ligeramente menor que L2. Random Forest obtuvo RMSE de validación 1.9681325361 °C, mayor que las alternativas lineales. Un algoritmo más complejo no garantizó una mejora en esta muestra.

La prueba reservada de L2 obtuvo RMSE 1.6536389952 °C, MAE 1.2131202137 °C y R² 0.9083958190. Esto no acredita exactitud en otras ubicaciones, meses o sensores independientes.

El modelo y el ensamblador se guardaron y recuperaron; la diferencia máxima entre sus predicciones originales y recuperadas fue 0.0.

### Reflexión técnica — borrador para revisar

Entrenar un solo modelo no permite saber si otra alternativa obtiene menor error.  
Comparé configuraciones usando el mismo bloque de validación para mantener un criterio común.  
L2 obtuvo el menor RMSE, pero la mejora frente a la base fue mínima.  
Random Forest tuvo mayor error, por lo que su complejidad no garantizó mejores resultados.  
Reservé fechas de prueba para evaluar la selección sin utilizarlas para elegirla.  
Las métricas de un mes no bastan para confiar en decisiones agrícolas operativas.  
Antes de usar el modelo necesitaría más periodos, datos independientes y un error aceptable definido.

### Estado de la entrega

La ejecución revisada corresponde a la actividad autónoma de AgroSmart. La práctica guiada de las tres fuentes de campo eléctrico y magnético sigue pendiente de sus archivos. También faltan los datos del estudiante, capturas, PDF y el anexo de feedback respondido personalmente. El borrador de reflexión debe leerse y ajustarse antes de la entrega.
